# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman123-123/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — ML Appendix, "What Predicts Health?" (Random Forest feature importance for health
score).** The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as
the top predictors of `health_score`, and to its credit flags this itself: *"the target itself is
partly constructed from some of these inputs, so importance is descriptive rather than causal."*
Worth pushing on: `health_score = impressions(30pts) + position(30pts) + ctr(20pts) +
scroll_depth(20pts)` by definition (see `docs/how-to-read-this-paper`, Health Score). Training a
model to predict a weighted sum FROM its own weighted inputs isn't really measuring "what predicts
health" — it's substantially re-deriving the composite formula's own weights. This is exactly
**leakage taxonomy #1 (label-derived features)** from `hunting-leakage-and-validating`: the label
was computed FROM these columns, and they're back in the feature set. The paper's own caveat is
the right instinct; I'd go one step further and say the importance ranking here isn't just
"descriptive" but expected-by-construction, and shouldn't be shown next to genuinely predictive
findings (like Finding B) without a clearer visual separation.

**Finding B — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy).**
This one predicts `trend_direction`-style growth/decline from `content_age`, `days_since_update`,
`days_visible`, `avg_position`, `word_count`, etc. — features that are NOT algebraically part of
the growth label, so this is a legitimate supervised question, closer to my own lane. My
methodology question is about the **split**, not the features: the Methodology page says "Random
Forest (80/20 split), Logistic Regression (80/20 split)" with no mention of grouping by brand —
and the portfolio spans 57 brands. My own w05/w06 work below shows that a plain random row split
on this exact kind of label inflates precision@50 from 0.74 (client-grouped, honest) to 0.94
(random split, naive) — a brand can "leak" its baseline traffic level across train/test the same
way a client does in my data. Without knowing whether the paper's 71% holdout accuracy came from a
grouped-by-brand split or a random row split, I can't tell how much of that number is genuine
signal versus brand memorization. That's not an accusation the paper is wrong — it's a question
the methodology section doesn't answer, and my own results below show why the answer matters.


In [1]:
# Both findings are read directly from docs/flyrank-seo-research-march-2026.pdf
# (ML Appendix pages: "What Predicts Health?" and "What Predicts Growth?", plus the
# Methodology page's "ML Pipeline" line). No code needed here -- this is a close reading
# of the paper cross-checked against docs/data-dictionary.md's health_score-equivalent
# construction rules and my own split-sensitivity result in section 2 below.
print("See markdown above: Finding A (health-score feature importance) and "
      "Finding B (growth-prediction holdout accuracy) with methodology questions.")


See markdown above: Finding A (health-score feature importance) and Finding B (growth-prediction holdout accuracy) with methodology questions.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I retrain the Week-5 winner (logistic regression, same leakage-safe features) two ways on the
identical starter CSV: a **naive random row split** (what Finding B's methodology page doesn't
rule out) versus my **grouped-by-client split** (what I actually use). Same label, same metric,
same K=50 — only the split changes.


In [2]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        os.system("git clone -q https://github.com/Eman123-123/flyrank-ml-internship.git")
    os.chdir("flyrank-ml-internship")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

raw_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
]
categorical_features = ["competition_level", "content_type", "main_intent"]


def add_features(frame):
    f = frame.copy()
    f["has_position"] = (f["avg_position"] > 0).astype(int)
    f["avg_position_clean"] = f["avg_position"].where(f["avg_position"] > 0, np.nan)
    for c in raw_numeric[5:]:
        f[f"log_{c}"] = np.log1p(f[c].clip(lower=0))
    f["has_keyword_data"] = f["search_volume"].notna().astype(int)
    f["has_word_count"] = f["word_count"].notna().astype(int)
    return f


log_numeric = [f"log_{c}" for c in raw_numeric[5:]]
final_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position_clean", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "has_position", "has_keyword_data", "has_word_count",
] + log_numeric


def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())


def fit_and_score(train_df, test_df, extra_cols=None):
    extra_cols = extra_cols or []
    train_p, test_p = add_features(train_df), add_features(test_df)
    feats = final_numeric + extra_cols + categorical_features
    X_train, y_train = train_p[feats], train_p["target_declining"]
    X_test, y_test = test_p[feats], test_p["target_declining"]
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
         final_numeric + extra_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    pipe = Pipeline([("prep", preprocess),
                      ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED))])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    return {
        "roc_auc": round(roc_auc_score(y_test, proba), 3),
        "avg_precision": round(average_precision_score(y_test, proba), 3),
        "precision_at_50": round(precision_at_k(y_test, proba, 50), 3),
        "n_test": len(y_test),
    }


# Grouped split (client holdout) -- what I actually use (same as w05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(df, groups=df["client_id"]))
grouped_result = fit_and_score(df.iloc[tr_idx].copy(), df.iloc[te_idx].copy())
grouped_result["split"] = "grouped_by_client (honest)"

# Naive random row split -- what a methodology page silent on grouping might mean
tr_idx2, te_idx2 = train_test_split(
    np.arange(len(df)), test_size=0.25, random_state=RANDOM_SEED, stratify=df["target_declining"]
)
random_result = fit_and_score(df.iloc[tr_idx2].copy(), df.iloc[te_idx2].copy())
random_result["split"] = "random_row_split (naive)"

split_comparison = pd.DataFrame([random_result, grouped_result])[
    ["split", "roc_auc", "avg_precision", "precision_at_50", "n_test"]
]
split_comparison


,split,roc_auc,avg_precision,precision_at_50,n_test
0,random_row_split (naive),0.716,0.733,0.94,7500
1,grouped_by_client (honest),0.625,0.622,0.74,7115


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Attack checklist from `hunting-leakage-and-validating`, on the exact features `w05_model.ipynb`
uses: timeline drawn (all features are trailing-90d aggregates or content properties, the label
is a 30d-vs-30d trend), no product flags used, base rate printed next to every metric (done in
w05), split grouped by client (section 2 above). The one deliberate test left is **train-with vs
train-without a suspect** — I add back `impressions_last_30d` / `impressions_prev_30d`, whose
difference *is* the label (`trend_pct`), and watch what happens on the identical grouped split.


In [3]:
suspect_cols = ["impressions_last_30d", "impressions_prev_30d"]

with_suspects = fit_and_score(df.iloc[tr_idx].copy(), df.iloc[te_idx].copy(), extra_cols=suspect_cols)

leakage_table = pd.DataFrame([
    {"features": "final leakage-safe set (w05)", "roc_auc": grouped_result["roc_auc"],
     "precision_at_50": grouped_result["precision_at_50"]},
    {"features": "+ impressions_last_30d / impressions_prev_30d", "roc_auc": with_suspects["roc_auc"],
     "precision_at_50": with_suspects["precision_at_50"]},
])
print(leakage_table.to_string(index=False))
print(
    f"\nAdding the suspects moves ROC AUC from {grouped_result['roc_auc']} to "
    f"{with_suspects['roc_auc']} and precision@50 from {grouped_result['precision_at_50']} to "
    f"{with_suspects['precision_at_50']} -- a jump to a near-perfect score is the confession "
    f"described in `hunting-leakage-and-validating`. Those two columns stay OUT of the real model."
)


                                     features  roc_auc  precision_at_50
                 final leakage-safe set (w05)    0.625             0.74
+ impressions_last_30d / impressions_prev_30d    0.846             1.00

Adding the suspects moves ROC AUC from 0.625 to 0.846 and precision@50 from 0.74 to 1.0 -- a jump to a near-perfect score is the confession described in `hunting-leakage-and-validating`. Those two columns stay OUT of the real model.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence I could have written (from w05):** *"Logistic regression predicts which pages
are declining, beating the baseline by 26 points of precision."*

**Rewritten, safe version:** *"On a client-held-out test split of the 30,000-row starter dataset,
logistic regression reached a measured precision@50 of 0.74 against a hand-rule baseline of 0.48
and a base rate of 0.517 — a directional lift that supports using the model's ranking as one input
to a human review queue. This is decision-support, not proof that any specific flagged page is
actually declining, and not evidence the model would generalize to clients or time periods outside
this snapshot."* The rewrite adds: the exact split (client-held-out, not just "test data"), the
base rate next to the headline number (per `hunting-leakage-and-validating`'s rule), and an
explicit "decision-support, not proof" boundary instead of the word "predicts" standing alone.


In [4]:
# No computation needed -- this section is a writing exercise. The two sentences above
# are printed here so the notebook's Run All includes the claim-rewrite artifact.
bold_claim = (
    "Logistic regression predicts which pages are declining, beating the baseline by "
    "26 points of precision."
)
safe_claim = (
    "On a client-held-out test split of the 30,000-row starter dataset, logistic regression "
    "reached a measured precision@50 of 0.74 against a hand-rule baseline of 0.48 and a base "
    "rate of 0.517 -- a directional lift that supports using the model's ranking as one input "
    "to a human review queue. This is decision-support, not proof that any specific flagged "
    "page is actually declining, and not evidence the model would generalize to clients or "
    "time periods outside this snapshot."
)
print("BOLD:", bold_claim)
print("\nSAFE:", safe_claim)


BOLD: Logistic regression predicts which pages are declining, beating the baseline by 26 points of precision.

SAFE: On a client-held-out test split of the 30,000-row starter dataset, logistic regression reached a measured precision@50 of 0.74 against a hand-rule baseline of 0.48 and a base rate of 0.517 -- a directional lift that supports using the model's ranking as one input to a human review queue. This is decision-support, not proof that any specific flagged page is actually declining, and not evidence the model would generalize to clients or time periods outside this snapshot.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.